# Model Test Notebook

This notebook gives one test example for each backend model using the exact input names each model expects.

In [ ]:
from types import SimpleNamespace

from Run_Model import run_model as run_distribution_model
from YAC_Model import run_model as run_yac_model
from Pass_Model_JP import predict_pass_probability

## Run Model Test

Expected fields for `Run_Model.run_model(play)`:
- `LOS`
- `goal_to_go`
- `down`
- `ydstogo`
- `formation` with values like `"shotgun"`, `"qbdropback"`, or `"both"`
- `run_location` with values `"left"`, `"middle"`, or `"right"`
- `run_gap` with values `"guard"`, `"tackle"`, or `"end"`

In [ ]:
run_play = SimpleNamespace(
    LOS=48,
    goal_to_go=0,
    down=2,
    ydstogo=5,
    formation="shotgun",
    run_location="right",
    run_gap="guard",
)

run_result = run_distribution_model(run_play)
run_result

## Pass -> YAC Flow

Your teammates are right that the pass model output can feed the YAC model.

### Variable passed from pass model into YAC model

- `pass_completion`

In this setup:
- `predict_pass_probability(...)` returns the probability that the pass is completed
- `YAC_Model.run_model(play)` expects a field named `pass_completion`
- so the pass model's output probability should be assigned to `play.pass_completion`

### Other values that should stay aligned between the two models

These are not outputs of the pass model, but they describe the same pass play and should usually be the same in both calls:
- `down`
- `ydstogo`
- `shotgun`
- `air_yards`
- `pass_location`
- offensive/defensive team context (`posteam`, `defteam`, `season` for YAC, and ranks for the pass model)

So the normal testing flow is:
1. call the pass model
2. store that probability
3. pass that probability into YAC as `pass_completion`

## Pass Completion Model Test

Expected arguments for `predict_pass_probability(...)`:
- `air_yards`
- `yardline_100`
- `ydstogo`
- `down`
- `qtr`
- `shotgun`
- `off_rank`
- `def_rank`
- `pass_location`

In [ ]:
# Shared pass-play inputs used by both the pass model and the YAC model.
pass_inputs = {
    "season": 2010,
    "posteam": "SEA",
    "defteam": "DEN",
    "LOS": 48,
    "goal_to_go": 0,
    "down": 2,
    "qtr": 2,
    "ydstogo": 5,
    "shotgun": 0,
    "air_yards": 5,
    "pass_location": "right",
    "off_rank": 8,
    "def_rank": 20,
}

yardline_100 = 110 - pass_inputs["LOS"]

pass_completion_probability = predict_pass_probability(
    air_yards=pass_inputs["air_yards"],
    yardline_100=yardline_100,
    ydstogo=pass_inputs["ydstogo"],
    down=pass_inputs["down"],
    qtr=pass_inputs["qtr"],
    shotgun=pass_inputs["shotgun"],
    off_rank=pass_inputs["off_rank"],
    def_rank=pass_inputs["def_rank"],
    pass_location=pass_inputs["pass_location"],
)

pass_completion_probability

## YAC Model Test Using Pass Model Output

Expected fields for `YAC_Model.run_model(play)`:
- `LOS`
- `goal_to_go`
- `down`
- `ydstogo`
- `posteam`
- `defteam`
- `season`
- `pass_location`
- `shotgun`
- `air_yards`
- `pass_completion`

Below, `pass_completion` is set equal to the pass model output from the previous cell.

In [ ]:
yac_play = SimpleNamespace(
    LOS=pass_inputs["LOS"],
    goal_to_go=pass_inputs["goal_to_go"],
    down=str(pass_inputs["down"]),
    ydstogo=pass_inputs["ydstogo"],
    posteam=pass_inputs["posteam"],
    defteam=pass_inputs["defteam"],
    season=pass_inputs["season"],
    pass_location=pass_inputs["pass_location"],
    shotgun=pass_inputs["shotgun"],
    air_yards=pass_inputs["air_yards"],
    pass_completion=float(pass_completion_probability),
)

yac_result = run_yac_model(yac_play)
yac_result